# 개인 계정 · Dunnhumby GCN 남은 8시드 → 10시드 집계

**seed49·50 재사용 → seed51 재개 → seed42~48 순차 실행**

- seed51 M1·M2 완료 결과 재사용, M3는 저장78epoch 다음79부터. M4·M5_A~D 새 학습.
- 완료18개 재사용, 1개 이어서 학습, 61개 새 학습. 모든80개가 모여야 10시드 평균·표본 표준편차를 출력합니다.
- 개인 계정으로 Drive를 연결하세요. 학교 계정에서는 실행하지 않습니다. 다른 실행과 seed를 중복 배정하지 마세요.
- 아래 입력 ZIP을 개인 **내 드라이브 최상위**에 한 번 업로드하세요: `dh_gcn_personal_resume_inputs.zip`.
- 고정 L4 / 300epoch / batch8192 / Adam lr0.0005 / uniform K1 / ID64·2층. DAY1~704 학습,705~711 신규상품 test, k1. 중간 test·조기종료·튜닝 없음.
- 입력은 로컬 디스크에서 한 번 준비. 학교 Drive의 전수조회·휴지통 정리는 사용하지 않습니다. 매epoch 동기 저장, 완료 checkpoint 압축 형태 전환은 기존 코드 그대로입니다. 저장공간은 실행 전 확인하고 여유를 확보하세요.
- 개인 Drive 안에 이름에 `_school_v1`이 들어간 seed51 폴더를 복원합니다. **원래 실행 신원을 유지하기 위한 이름**이며 학교 Drive를 쓰는 것이 아닙니다.
- 노트북 전체 실행은 실제 데이터/GPU가 필요합니다. 로컬에서는 구조·원본 무결성·재사용 경로를 검사했습니다. 최초 재개 로그 `epoch 78에서 재개`를 확인하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, zipfile, subprocess, torch
drive.mount('/content/drive')
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), '기존 조건과 같은 L4 GPU를 선택하세요.'
BUNDLE=Path('/content/drive/MyDrive/dh_gcn_personal_resume_inputs.zip')
assert BUNDLE.is_file(), '제공한 입력 ZIP을 개인 내 드라이브 최상위에 올려주세요.'
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''): h.update(b)
    return h.hexdigest()
assert sha(BUNDLE)=='eee4b2b5f1d969534761f0b79a032af21d6f7491e95c0fce9369514b6dcd8af2', '입력 ZIP이 제공한 원본과 다릅니다.'
ASSETS=Path('/content/gcn_personal_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    name='clv_dh_gcn_personal_remaining.py'
    (ASSETS/name).write_bytes(z.read(name))
sys.path.insert(0,str(ASSETS))
import clv_dh_gcn_personal_remaining as task
task.unpack_verified(BUNDLE,ASSETS)
SOURCE=task.SOURCE
REPO=Path('/content/gcn-frozen-'+SOURCE[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE
os.chdir(REPO)
sys.path.insert(0,str(REPO))
print('입력·백업 검증 완료 / GPU:',torch.cuda.get_device_name(0))

## 1. 입력 준비·원본 설계 대조 (학습 없음)
재연결 후에는 위에서부터 실행합니다. 개인 Drive에 이미 저장된 진행 파일은 덮어쓰지 않습니다.

In [ ]:
ctx=task.prepare(ASSETS,'/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_personal_final10_v1')
print(json.dumps(task.report(ctx)[0],ensure_ascii=False,indent=2))

## 2. 남은 학습 실행
매epoch 저장합니다. 완료 모형은 재학습·재평가하지 않습니다. 코드/설정을 바꾸지 말고 재개하세요. 오류 발생 시 원인을 확인한 뒤 동일 셀을 재실행합니다. 저장 실패를 무시하지 않습니다.

In [ ]:
status=task.run(ctx)
print(json.dumps(status,ensure_ascii=False,indent=2))

## 3. 결과 회수 (완료·중단 후 실행)
미완료는 partial ZIP으로 명시합니다. 체크포인트는 Drive에 유지하며 결과 ZIP에는 넣지 않습니다. 평균/표준편차는 80개 모두 완료 후 생성합니다. 유의성 검정은 별도입니다.

In [ ]:
from google.colab import files
output=task.export(ctx)
print('Drive 결과:',output)
files.download(output)